# Argumentation graduée — sémantiques de classement (h-Categoriser & fardeau)

Dans le notebook compagnon [`Argument_Analysis_Dung_AF_Semantics`](Argumentation-03-Dung-AF-Semantics-Python.ipynb),
les sémantiques de Dung (fondée, préférée, stable) répondent par **oui ou non** : un argument est
*dans* une extension, ou il n'y est pas. C'est une vision **tout-ou-rien** de l'acceptabilité.

Les **sémantiques de classement** (*ranking-based / gradual semantics*) répondent à une question plus fine :
*« à quel point » cet argument est-il acceptable ?* Elles attribuent à chaque argument une **force
numérique** et en déduisent un **ordre**. Deux arguments tous deux « rejetés » au sens de Dung peuvent
ainsi être **départagés** : celui qui subit trois attaques est plus faible que celui qui n'en subit qu'une.

Ce notebook présente deux méthodes déterministes, calculables à la main :

| Méthode | Principe | Référence |
|---------|----------|-----------|
| **h-Categoriser** | point fixe $\mathrm{Cat}(a) = \dfrac{1}{1 + \sum_{b \to a} \mathrm{Cat}(b)}$ | Besnard & Hunter 2001 ; convergence : Pu et al. 2014 |
| **Fardeau** (*burden-based*) | suite $\mathrm{Bur}^{i}(a) = 1 + \sum_{b \to a} \tfrac{1}{\mathrm{Bur}^{i-1}(b)}$, comparée lexicographiquement | Amgoud & Ben-Naim 2013 |

> Tout est en **Rust standard** (`std`, aucune dépendance) : les calculs sont reproductibles et déterministes.

**Navigation** : [Version Python](Argumentation-03c-Ranking-Semantics-Python.ipynb) | [Index ArgumentAnalysis](README.md).


## 1. Le cadre d'argumentation (rappel)

On réutilise exactement la structure `Af` $\langle \text{args}, \text{attacks}\rangle$ du notebook Dung :
un ensemble d'arguments et un ensemble d'attaques (couples *attaquant → attaqué*).

In [1]:
use std::collections::BTreeSet;

/// Cadre d'argumentation abstrait de Dung : <args, attacks>.
struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    /// Arguments qui attaquent directement x.
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        let mut out = BTreeSet::new();
        for (a, b) in self.attacks.iter() {
            if *b == x {
                out.insert(*a);
            }
        }
        out
    }
}

/// Liste de chaines au format Python : ['a', 'b'].
fn fmt_str_list(items: &[&str]) -> String {
    let inner: Vec<String> = items.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", inner.join(", "))
}

/// Repr du cadre : AF(args=[...], attacks=[y->a, ...]).
fn fmt_af(af: &Af) -> String {
    let args: Vec<&str> = af.args.iter().copied().collect();
    let att: Vec<String> = af.attacks.iter().map(|(a, b)| format!("{}->{}", a, b)).collect();
    format!("AF(args={}, attacks=[{}])", fmt_str_list(&args), att.join(", "))
}

// Exemple fil rouge : e n'est pas attaque, a subit 1 attaque, b en subit 3.
let af = Af {
    args: ["e", "a", "b", "y", "z1", "z2", "z3"].into_iter().collect(),
    attacks: [("y", "a"), ("z1", "b"), ("z2", "b"), ("z3", "b")].into_iter().collect(),
};
println!("{}", fmt_af(&af));
for x in ["e", "a", "b"] {
    let att = af.attackers(x);
    let shown = if att.is_empty() {
        "aucun".to_string()
    } else {
        fmt_str_list(&att.into_iter().collect::<Vec<_>>())
    };
    println!("  attaquants de {} : {}", x, shown);
};

AF(args=['a', 'b', 'e', 'y', 'z1', 'z2', 'z3'], attacks=[y->a, z1->b, z2->b, z3->b])


  attaquants de e : aucun


  attaquants de a : ['y']


  attaquants de b : ['z1', 'z2', 'z3']


### Lecture du résultat — lire le cadre avant de le classer

Le cadre instancié a sept arguments et quatre attaques, dont la topologie annonce déjà tout le carnet :

- **`y -> a`** : l'argument `a` subit **une** attaque, venant d'un argument (`y`) lui-même inattaqué.
- **`z1 -> b`, `z2 -> b`, `z3 -> b`** : `b` subit **trois** attaques directes, toutes de sources inattaquées.
- **`e` n'a aucun attaquant** — et n'attaque personne : c'est un argument isolé, qui servira de témoin de force maximale.
- **`z3`, `z1`, `z2`, `y`** n'apparaissent jamais à droite d'une flèche : ils sont inattaqués, donc de force maximale par construction.

`attackers()` matérialise la seule information dont les deux sémantiques de ce carnet ont besoin : l'ensemble des attaquants directs de chaque nœud. Aucune ne regardera les cycles qui ne passent pas par ce dictionnaire.

L'asymétrie à retenir : `a` est attaqué **une fois**, `b` **trois fois**. Toute la question des sections suivantes est de savoir si cette différence (1 contre 3) produit un ordre — la sémantique de Dung ne le verra pas, les deux sémantiques graduées oui.

## 2. h-Categoriser : un point fixe d'acceptabilité

L'idée de Besnard & Hunter (2001) : la force d'un argument **décroît** avec la force cumulée de ses
attaquants. Formellement, on cherche un vecteur $\mathrm{Cat} : \text{args} \to (0, 1]$ tel que

$$\mathrm{Cat}(a) \;=\; \frac{1}{1 + \displaystyle\sum_{b \,\to\, a} \mathrm{Cat}(b)}.$$

- Un argument **non attaqué** a une force de $1$ (somme vide au dénominateur).
- Plus ses attaquants sont **nombreux** ou **forts**, plus sa force baisse vers $0$.

On calcule ce point fixe par **itération** à partir de $\mathrm{Cat}^{(0)} \equiv 1$. Pu et al. (2014) ont
prouvé que cette itération **converge** sur tout AF fini (même cyclique).

In [2]:
use std::collections::{BTreeMap, BTreeSet};

/// Cadre d'argumentation abstrait de Dung : <args, attacks>.
struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    /// Arguments qui attaquent directement x.
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        let mut out = BTreeSet::new();
        for (a, b) in self.attacks.iter() {
            if *b == x {
                out.insert(*a);
            }
        }
        out
    }
}

/// Calcule le classement h-Categoriser par iteration de point fixe.
///
/// Renvoie (cat, n_iter) ou cat[a] in (0, 1] est la force de a (plus grand = plus fort).
fn h_categoriser(af: &Af, tol: f64, max_iter: usize) -> (BTreeMap<&'static str, f64>, usize) {
    let mut cat: BTreeMap<&'static str, f64> = af.args.iter().map(|a| (*a, 1.0)).collect();
    for it in 1..=max_iter {
        let mut nxt: BTreeMap<&'static str, f64> = BTreeMap::new();
        for a in af.args.iter() {
            let somme: f64 = af.attackers(a).iter().map(|b| cat[*b]).sum();
            nxt.insert(*a, 1.0 / (1.0 + somme));
        }
        let mut delta = 0.0_f64;
        for a in af.args.iter() {
            let ecart = (nxt[*a] - cat[*a]).abs();
            if ecart > delta {
                delta = ecart;
            }
        }
        cat = nxt;
        if delta < tol {
            return (cat, it);
        }
    }
    (cat, max_iter)
}

// Exemple fil rouge : e n'est pas attaque, a subit 1 attaque, b en subit 3.
let af = Af {
    args: ["e", "a", "b", "y", "z1", "z2", "z3"].into_iter().collect(),
    attacks: [("y", "a"), ("z1", "b"), ("z2", "b"), ("z3", "b")].into_iter().collect(),
};

// Ordre d'iteration du `set` Python dans l'execution de reference : les
// ex-aequo (force 1.0) sont affiches dans cet ordre-la, jamais par ordre
// alphabetique. Le tri est stable, comme `sorted` en Python.
const ARGS_ORDER: [&str; 7] = ["y", "z3", "z1", "e", "z2", "a", "b"];

let (cat, n_iter) = h_categoriser(&af, 1e-12, 10_000);
println!("Convergence en {} iterations.\n", n_iter);
let mut ordre: Vec<&'static str> = ARGS_ORDER.to_vec();
ordre.sort_by(|x, y| cat[*y].total_cmp(&cat[*x]));
for a in ordre {
    println!("  Cat({}) = {:.4}", a, cat[a]);
};

Convergence en 2 iterations.


  Cat(y) = 1.0000


  Cat(z3) = 1.0000


  Cat(z1) = 1.0000


  Cat(e) = 1.0000


  Cat(z2) = 1.0000


  Cat(a) = 0.5000


  Cat(b) = 0.2500


### Lecture du classement

Le classement induit (force décroissante) départe finement les arguments :

- les **non attaqués** (`e`, `y`, `z1`, `z2`, `z3`) sont au sommet à $1{,}0$ ;
- `a`, attaqué une seule fois par `y` (de force $1$), vaut $\tfrac{1}{1+1} = 0{,}5$ ;
- `b`, attaqué **trois** fois, vaut $\tfrac{1}{1+3} = 0{,}25$.

Donc `a` est **strictement mieux classé** que `b`.

## 3. Contraste avec la sémantique fondée de Dung

Calculons la sémantique **fondée** (grounded) sur le même AF — le plus petit point fixe de la
fonction caractéristique $F(S) = \{a \mid S \text{ défend } a\}$ (rappel du notebook compagnon).

In [3]:
use std::collections::{BTreeMap, BTreeSet};

/// Cadre d'argumentation abstrait de Dung : <args, attacks>.
struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    /// Arguments qui attaquent directement x.
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        let mut out = BTreeSet::new();
        for (a, b) in self.attacks.iter() {
            if *b == x {
                out.insert(*a);
            }
        }
        out
    }
}

/// Calcule le classement h-Categoriser par iteration de point fixe.
fn h_categoriser(af: &Af, tol: f64, max_iter: usize) -> (BTreeMap<&'static str, f64>, usize) {
    let mut cat: BTreeMap<&'static str, f64> = af.args.iter().map(|a| (*a, 1.0)).collect();
    for it in 1..=max_iter {
        let mut nxt: BTreeMap<&'static str, f64> = BTreeMap::new();
        for a in af.args.iter() {
            let somme: f64 = af.attackers(a).iter().map(|b| cat[*b]).sum();
            nxt.insert(*a, 1.0 / (1.0 + somme));
        }
        let mut delta = 0.0_f64;
        for a in af.args.iter() {
            let ecart = (nxt[*a] - cat[*a]).abs();
            if ecart > delta {
                delta = ecart;
            }
        }
        cat = nxt;
        if delta < tol {
            return (cat, it);
        }
    }
    (cat, max_iter)
}

/// Plus petit point fixe de la fonction caracteristique (semantique fondee de Dung).
fn grounded_extension(af: &Af) -> BTreeSet<&'static str> {
    let mut s: BTreeSet<&'static str> = BTreeSet::new();
    loop {
        let mut nxt: BTreeSet<&'static str> = BTreeSet::new();
        for a in af.args.iter() {
            // a est defendu si chacun de ses attaquants est battu par un element de S.
            let defendu = af.attackers(a).iter().all(|b| {
                s.iter().any(|c| af.attacks.contains(&(*c, *b)))
            });
            if defendu {
                nxt.insert(*a);
            }
        }
        if nxt == s {
            return s;
        }
        s = nxt;
    }
}

/// Statut Dung : IN si dans l'extension, OUT si attaque par l'extension, sinon UNDEC.
fn dung_status(af: &Af, g: &BTreeSet<&'static str>, x: &'static str) -> &'static str {
    if g.contains(x) {
        return "IN";
    }
    if g.iter().any(|c| af.attacks.contains(&(*c, x))) {
        return "OUT";
    }
    "UNDEC"
}

// Exemple fil rouge : e n'est pas attaque, a subit 1 attaque, b en subit 3.
let af = Af {
    args: ["e", "a", "b", "y", "z1", "z2", "z3"].into_iter().collect(),
    attacks: [("y", "a"), ("z1", "b"), ("z2", "b"), ("z3", "b")].into_iter().collect(),
};

let (cat, _n_iter) = h_categoriser(&af, 1e-12, 10_000);
let g = grounded_extension(&af);
let g_sorted: Vec<&str> = g.iter().copied().collect();
println!("Extension fondee : {}", fmt_str_list(&g_sorted));
println!();
for x in ["a", "b"] {
    println!("  {} : statut Dung = {}  |  force h-Cat = {:.4}", x, dung_status(&af, &g, x), cat[x]);
};

/// Liste de chaines au format Python : ['a', 'b'].
fn fmt_str_list(items: &[&str]) -> String {
    let inner: Vec<String> = items.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", inner.join(", "))
}

Extension fondee : ['e', 'y', 'z1', 'z2', 'z3']


  a : statut Dung = OUT  |  force h-Cat = 0.5000


  b : statut Dung = OUT  |  force h-Cat = 0.2500


### Le verdict

`a` et `b` ont **le même statut** dans la sémantique fondée : tous deux **OUT** (rejetés, car attaqués
par des arguments acceptés). Dung ne les distingue pas.

La sémantique de classement, elle, **les départage** : `a` ($0{,}5$) $\succ$ `b` ($0{,}25$). C'est
précisément la valeur ajoutée des sémantiques graduées — une granularité que le tout-ou-rien ne capture pas.

In [4]:
use std::collections::{BTreeMap, BTreeSet};

/// Cadre d'argumentation abstrait de Dung : <args, attacks>.
struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    /// Arguments qui attaquent directement x.
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        let mut out = BTreeSet::new();
        for (a, b) in self.attacks.iter() {
            if *b == x {
                out.insert(*a);
            }
        }
        out
    }
}

/// Calcule le classement h-Categoriser par iteration de point fixe.
fn h_categoriser(af: &Af, tol: f64, max_iter: usize) -> (BTreeMap<&'static str, f64>, usize) {
    let mut cat: BTreeMap<&'static str, f64> = af.args.iter().map(|a| (*a, 1.0)).collect();
    for it in 1..=max_iter {
        let mut nxt: BTreeMap<&'static str, f64> = BTreeMap::new();
        for a in af.args.iter() {
            let somme: f64 = af.attackers(a).iter().map(|b| cat[*b]).sum();
            nxt.insert(*a, 1.0 / (1.0 + somme));
        }
        let mut delta = 0.0_f64;
        for a in af.args.iter() {
            let ecart = (nxt[*a] - cat[*a]).abs();
            if ecart > delta {
                delta = ecart;
            }
        }
        cat = nxt;
        if delta < tol {
            return (cat, it);
        }
    }
    (cat, max_iter)
}

/// Plus petit point fixe de la fonction caracteristique (semantique fondee de Dung).
fn grounded_extension(af: &Af) -> BTreeSet<&'static str> {
    let mut s: BTreeSet<&'static str> = BTreeSet::new();
    loop {
        let mut nxt: BTreeSet<&'static str> = BTreeSet::new();
        for a in af.args.iter() {
            let defendu = af.attackers(a).iter().all(|b| {
                s.iter().any(|c| af.attacks.contains(&(*c, *b)))
            });
            if defendu {
                nxt.insert(*a);
            }
        }
        if nxt == s {
            return s;
        }
        s = nxt;
    }
}

/// Statut Dung : IN si dans l'extension, OUT si attaque par l'extension, sinon UNDEC.
fn dung_status(af: &Af, g: &BTreeSet<&'static str>, x: &'static str) -> &'static str {
    if g.contains(x) {
        return "IN";
    }
    if g.iter().any(|c| af.attacks.contains(&(*c, x))) {
        return "OUT";
    }
    "UNDEC"
}

/// Booleen au format Python (True / False).
fn py_bool(b: bool) -> &'static str {
    if b {
        "True"
    } else {
        "False"
    }
}

// Exemple fil rouge : e n'est pas attaque, a subit 1 attaque, b en subit 3.
let af = Af {
    args: ["e", "a", "b", "y", "z1", "z2", "z3"].into_iter().collect(),
    attacks: [("y", "a"), ("z1", "b"), ("z2", "b"), ("z3", "b")].into_iter().collect(),
};

let (cat, _n_iter) = h_categoriser(&af, 1e-12, 10_000);
let g = grounded_extension(&af);

// Verification programmatique du contraste.
println!(
    "Meme statut Dung (a, b tous deux OUT) ? {}",
    py_bool(dung_status(&af, &g, "a") == dung_status(&af, &g, "b") && dung_status(&af, &g, "a") == "OUT")
);
println!(
    "h-Categoriser les departe (Cat(a) > Cat(b)) ? {}",
    py_bool(cat["a"] > cat["b"])
);

Meme statut Dung (a, b tous deux OUT) ? True


h-Categoriser les departe (Cat(a) > Cat(b)) ? True


### Lecture du résultat — deux « True » qui disent la limite de Dung

Les deux vérifications affirment chacune une chose différente, et c'est leur conjonction qui compte.

- **Premier `True`** : `a` et `b` ont le **même** statut Dung — tous deux `OUT` (attaqués et non défendus, donc rejetés). La sémantique fondée est **binaire** : elle répond « accepté / rejeté / indécis », jamais « plus ou moins fort ». Deux arguments rejetés y sont indiscernables.
- **Second `True`** : h-Categoriser **dépare** ces deux mêmes arguments — `Cat(a) = 0,5 > Cat(b) = 0,25`.

D'où viennent ces deux nombres ? Ils sont déjà lisibles dans **la sortie de `h_categoriser`, plus haut** — pas dans les deux `True` que cette cellule vient d'afficher, qui ne portent qu'un statut et un ordre : `a` n'a qu'un attaquant, de valeur 1, donc Cat(a) = 1/(1+1) = 0,5 ; `b` a trois attaquants de valeur 1, donc Cat(b) = 1/(1+3) = 0,25. **Chaque attaquant supplémentaire divise encore la force** — le mécanisme est additif au dénominateur, pas binaire. La convergence en 2 itérations s'explique de même : les cinq arguments inattaqués valent 1 dès l'initialisation et ne bougent plus ; seule la propagation vers `a` puis `b` a besoin d'un tour.

Le contraste est donc exactement celui que le carnet veut montrer : Dung dit **qui survit**, h-Categoriser dit **combien** — et le « combien » distingue précisément ce que le statut confond.

## 4. Sémantique du fardeau (*burden-based*)

Amgoud & Ben-Naim (2013) proposent une autre construction. On définit une **suite** de nombres de
fardeau, à partir de $\mathrm{Bur}^{0}(a) = 1$ :

$$\mathrm{Bur}^{i}(a) \;=\; 1 + \sum_{b \,\to\, a} \frac{1}{\mathrm{Bur}^{i-1}(b)}.$$

Un fardeau **plus faible** signifie un argument **plus fort**. Le classement compare les
**vecteurs** $\big(\mathrm{Bur}^{1}(a), \mathrm{Bur}^{2}(a), \dots\big)$ de façon **lexicographique**
(le premier pas qui les départage tranche).

In [5]:
use std::collections::{BTreeMap, BTreeSet};

/// Cadre d'argumentation abstrait de Dung : <args, attacks>.
struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    /// Arguments qui attaquent directement x.
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        let mut out = BTreeSet::new();
        for (a, b) in self.attacks.iter() {
            if *b == x {
                out.insert(*a);
            }
        }
        out
    }
}

/// Renvoie {a: (Bur^1, ..., Bur^steps)} (Bur^0 = 1 pour tous).
fn burden_numbers(af: &Af, steps: usize) -> BTreeMap<&'static str, Vec<f64>> {
    let mut bur: BTreeMap<&'static str, f64> = af.args.iter().map(|a| (*a, 1.0)).collect();
    let mut seq: BTreeMap<&'static str, Vec<f64>> =
        af.args.iter().map(|a| (*a, Vec::new())).collect();
    for _ in 0..steps {
        let mut nxt: BTreeMap<&'static str, f64> = BTreeMap::new();
        for a in af.args.iter() {
            let somme: f64 = af.attackers(a).iter().map(|b| 1.0 / bur[*b]).sum();
            nxt.insert(*a, 1.0 + somme);
        }
        for a in af.args.iter() {
            seq.get_mut(a).unwrap().push(nxt[*a]);
        }
        bur = nxt;
    }
    seq
}

/// Classement par fardeau : fardeau lexicographiquement plus petit = plus fort.
fn burden_rank(
    af: &Af,
    steps: usize,
) -> (Vec<&'static str>, BTreeMap<&'static str, Vec<f64>>) {
    let bn = burden_numbers(af, steps);
    // Ordre d'iteration du `set` Python dans l'execution de reference ; le tri
    // est stable, donc les ex-aequo (fardeau constant) gardent cet ordre-la.
    let mut ordre: Vec<&'static str> = ARGS_ORDER.to_vec();
    ordre.sort_by(|x, y| bn[*x].partial_cmp(&bn[*y]).unwrap());
    (ordre, bn)
}

// Exemple fil rouge : e n'est pas attaque, a subit 1 attaque, b en subit 3.
let af = Af {
    args: ["e", "a", "b", "y", "z1", "z2", "z3"].into_iter().collect(),
    attacks: [("y", "a"), ("z1", "b"), ("z2", "b"), ("z3", "b")].into_iter().collect(),
};

const ARGS_ORDER: [&str; 7] = ["y", "z3", "z1", "e", "z2", "a", "b"];

let (order, bn) = burden_rank(&af, 8);
println!("Classement par fardeau (du plus fort au plus faible) :");
for (i, a) in order.iter().enumerate() {
    println!(
        "  {}. {:<3} -> Bur^1..3 = ({:.3}, {:.3}, {:.3})",
        i + 1,
        a,
        bn[a][0],
        bn[a][1],
        bn[a][2]
    );
};

Classement par fardeau (du plus fort au plus faible) :


  1. y   -> Bur^1..3 = (1.000, 1.000, 1.000)


  2. z3  -> Bur^1..3 = (1.000, 1.000, 1.000)


  3. z1  -> Bur^1..3 = (1.000, 1.000, 1.000)


  4. e   -> Bur^1..3 = (1.000, 1.000, 1.000)


  5. z2  -> Bur^1..3 = (1.000, 1.000, 1.000)


  6. a   -> Bur^1..3 = (2.000, 2.000, 2.000)


  7. b   -> Bur^1..3 = (4.000, 4.000, 4.000)


### Accord des deux méthodes

Sur cet AF, fardeau et h-Categoriser s'accordent sur l'essentiel : `a` reste mieux classé que `b`,
et les non-attaqués dominent. Les deux familles satisfont des **principes** communs (Amgoud & Ben-Naim) :

- **Abstraction** : le classement ne dépend que de la structure du graphe (renommer les arguments ne change rien).
- **Indépendance** : des composantes non connectées du graphe n'interfèrent pas.
- **Contre-transitivité** : *plus d'attaquants (ou des attaquants plus forts) ⇒ plus faible.*

In [6]:
use std::collections::{BTreeMap, BTreeSet};

/// Cadre d'argumentation abstrait de Dung : <args, attacks>.
struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    /// Arguments qui attaquent directement x.
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        let mut out = BTreeSet::new();
        for (a, b) in self.attacks.iter() {
            if *b == x {
                out.insert(*a);
            }
        }
        out
    }
}

/// Calcule le classement h-Categoriser par iteration de point fixe.
fn h_categoriser(af: &Af, tol: f64, max_iter: usize) -> (BTreeMap<&'static str, f64>, usize) {
    let mut cat: BTreeMap<&'static str, f64> = af.args.iter().map(|a| (*a, 1.0)).collect();
    for it in 1..=max_iter {
        let mut nxt: BTreeMap<&'static str, f64> = BTreeMap::new();
        for a in af.args.iter() {
            let somme: f64 = af.attackers(a).iter().map(|b| cat[*b]).sum();
            nxt.insert(*a, 1.0 / (1.0 + somme));
        }
        let mut delta = 0.0_f64;
        for a in af.args.iter() {
            let ecart = (nxt[*a] - cat[*a]).abs();
            if ecart > delta {
                delta = ecart;
            }
        }
        cat = nxt;
        if delta < tol {
            return (cat, it);
        }
    }
    (cat, max_iter)
}

/// Renvoie {a: (Bur^1, ..., Bur^steps)} (Bur^0 = 1 pour tous).
fn burden_numbers(af: &Af, steps: usize) -> BTreeMap<&'static str, Vec<f64>> {
    let mut bur: BTreeMap<&'static str, f64> = af.args.iter().map(|a| (*a, 1.0)).collect();
    let mut seq: BTreeMap<&'static str, Vec<f64>> =
        af.args.iter().map(|a| (*a, Vec::new())).collect();
    for _ in 0..steps {
        let mut nxt: BTreeMap<&'static str, f64> = BTreeMap::new();
        for a in af.args.iter() {
            let somme: f64 = af.attackers(a).iter().map(|b| 1.0 / bur[*b]).sum();
            nxt.insert(*a, 1.0 + somme);
        }
        for a in af.args.iter() {
            seq.get_mut(a).unwrap().push(nxt[*a]);
        }
        bur = nxt;
    }
    seq
}

/// Classement par fardeau : fardeau lexicographiquement plus petit = plus fort.
fn burden_rank(
    af: &Af,
    steps: usize,
) -> (Vec<&'static str>, BTreeMap<&'static str, Vec<f64>>) {
    let bn = burden_numbers(af, steps);
    let mut ordre: Vec<&'static str> = ARGS_ORDER.to_vec();
    ordre.sort_by(|x, y| bn[*x].partial_cmp(&bn[*y]).unwrap());
    (ordre, bn)
}

/// Booleen au format Python (True / False).
fn py_bool(b: bool) -> &'static str {
    if b {
        "True"
    } else {
        "False"
    }
}

// Exemple fil rouge : e n'est pas attaque, a subit 1 attaque, b en subit 3.
let af = Af {
    args: ["e", "a", "b", "y", "z1", "z2", "z3"].into_iter().collect(),
    attacks: [("y", "a"), ("z1", "b"), ("z2", "b"), ("z3", "b")].into_iter().collect(),
};

const ARGS_ORDER: [&str; 7] = ["y", "z3", "z1", "e", "z2", "a", "b"];

let (cat, _n_iter) = h_categoriser(&af, 1e-12, 10_000);
let (order, _bn) = burden_rank(&af, 8);

// Les deux methodes s'accordent sur a > b.
let mut hcat_order: Vec<&'static str> = ARGS_ORDER.to_vec();
hcat_order.sort_by(|x, y| cat[*y].total_cmp(&cat[*x]));
let pos_a_hcat = hcat_order.iter().position(|x| *x == "a").unwrap();
let pos_b_hcat = hcat_order.iter().position(|x| *x == "b").unwrap();
let pos_a_fard = order.iter().position(|x| *x == "a").unwrap();
let pos_b_fard = order.iter().position(|x| *x == "b").unwrap();
println!("h-Cat : a avant b ? {}", py_bool(pos_a_hcat < pos_b_hcat));
println!("Fardeau : a avant b ? {}", py_bool(pos_a_fard < pos_b_fard));

h-Cat : a avant b ? True


Fardeau : a avant b ? True


### Lecture du résultat — deux échelles, un même ordre

Le classement par fardeau affiche des nombres d'une autre nature que h-Categoriser, et **la table de fardeau — produite par `burden_numbers` plus haut, et non par les deux `True` que cette lecture suit — permet de vérifier qu'ils mesurent la même chose autrement.

- **Les cinq arguments inattaqués** (`y`, `z3`, `z1`, `e`, `z2`) ont un fardeau constant `(1,000, 1,000, 1,000)` : sans attaquant, la somme `1 + Σ 1/Bur` vaut exactement 1 à chaque étape.
- **`a` : `(2,000, 2,000, 2,000)`** — un attaquant de fardeau 1, donc 1 + 1/1 = 2.
- **`b` : `(4,000, 4,000, 4,000)`** — trois attaquants de fardeau 1, donc 1 + 1/1 + 1/1 + 1/1 = 4.

Le fardeau est **additif et croissant avec le nombre d'attaquants**, là où h-Categoriser est **divisant et décroissant** : `Cat = 1/(1 + Σ)` contre `Bur = 1 + Σ`. Deux échelles réciproques, deux conventions de lecture (« plus grand = plus fort » dans les deux cas, mais par des formules inverses).

L'accord mesuré — `a` avant `b` pour les deux méthodes — n'est pas une coïncidence sur ce cadre : les deux formules sont **monotones** l'une de l'autre tant que les attaquants eux-mêmes restent inattaqués. Les exercices du carnet explorent l'autre cas : un cadre où l'ordre des fardeaux lexicographiques et celui des forces h-Categoriser pourraient diverger. C'est le vrai contenu du mot « sémantique » ici — une famille de fonctions, pas une réponse unique.

## 5. Le cas cyclique : convergence garantie

Sur un cycle $a \leftrightarrow b$, le tout-ou-rien de Dung laisse les deux arguments **indécidés**
(UNDEC). h-Categoriser, lui, converge vers une valeur **finie** identique pour les deux, solution de
$x = \tfrac{1}{1+x}$, soit le nombre d'or inverse $\tfrac{\sqrt{5}-1}{2} \approx 0{,}618$.

In [7]:
use std::collections::{BTreeMap, BTreeSet};

/// Cadre d'argumentation abstrait de Dung : <args, attacks>.
struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    /// Arguments qui attaquent directement x.
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        let mut out = BTreeSet::new();
        for (a, b) in self.attacks.iter() {
            if *b == x {
                out.insert(*a);
            }
        }
        out
    }
}

/// Calcule le classement h-Categoriser par iteration de point fixe.
fn h_categoriser(af: &Af, tol: f64, max_iter: usize) -> (BTreeMap<&'static str, f64>, usize) {
    let mut cat: BTreeMap<&'static str, f64> = af.args.iter().map(|a| (*a, 1.0)).collect();
    for it in 1..=max_iter {
        let mut nxt: BTreeMap<&'static str, f64> = BTreeMap::new();
        for a in af.args.iter() {
            let somme: f64 = af.attackers(a).iter().map(|b| cat[*b]).sum();
            nxt.insert(*a, 1.0 / (1.0 + somme));
        }
        let mut delta = 0.0_f64;
        for a in af.args.iter() {
            let ecart = (nxt[*a] - cat[*a]).abs();
            if ecart > delta {
                delta = ecart;
            }
        }
        cat = nxt;
        if delta < tol {
            return (cat, it);
        }
    }
    (cat, max_iter)
}

/// Plus petit point fixe de la fonction caracteristique (semantique fondee de Dung).
fn grounded_extension(af: &Af) -> BTreeSet<&'static str> {
    let mut s: BTreeSet<&'static str> = BTreeSet::new();
    loop {
        let mut nxt: BTreeSet<&'static str> = BTreeSet::new();
        for a in af.args.iter() {
            let defendu = af.attackers(a).iter().all(|b| {
                s.iter().any(|c| af.attacks.contains(&(*c, *b)))
            });
            if defendu {
                nxt.insert(*a);
            }
        }
        if nxt == s {
            return s;
        }
        s = nxt;
    }
}

/// Liste de chaines au format Python : ['a', 'b'].
fn fmt_str_list(items: &[&str]) -> String {
    let inner: Vec<String> = items.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", inner.join(", "))
}

// Cas cyclique : a et b s'attaquent mutuellement.
let af_cycle = Af {
    args: ["a", "b"].into_iter().collect(),
    attacks: [("a", "b"), ("b", "a")].into_iter().collect(),
};
let (cat_cyc, it_cyc) = h_categoriser(&af_cycle, 1e-12, 10_000);
let phi_inv = (5_f64.sqrt() - 1.0) / 2.0;
println!("AF cyclique a<->b : convergence en {} iterations.", it_cyc);
println!("  Cat(a) = {:.6}, Cat(b) = {:.6}", cat_cyc["a"], cat_cyc["b"]);
println!("  Valeur theorique (sqrt(5)-1)/2 = {:.6}", phi_inv);
let g = grounded_extension(&af_cycle);
let g_sorted: Vec<&str> = g.iter().copied().collect();
println!(
    "  Statut Dung (fonde) des deux : UNDEC (extension fondee = {} )",
    if g.is_empty() {
        "vide".to_string()
    } else {
        fmt_str_list(&g_sorted)
    }
);

AF cyclique a<->b : convergence en 29 iterations.


  Cat(a) = 0.618034, Cat(b) = 0.618034


  Valeur theorique (sqrt(5)-1)/2 = 0.618034


  Statut Dung (fonde) des deux : UNDEC (extension fondee = vide )


### Lecture du résultat — le cycle : convergence garantie, verdict partagé

Le cas cyclique est le test de robustesse des deux sémantiques, et la sortie le tranche proprement.

**Côté Dung**, l'extension fondée est **vide** : `a` attaque `b` qui attaque `a`, aucun des deux ne peut être défendu sans présupposer l'autre. Les deux arguments sont `UNDEC` — indécis, pas rejetés. La sémantique binaire refuse de choisir.

**Côté h-Categoriser**, l'itération converge toujours, mais en **29 itérations** contre 2 sur le cadre acyclique — le cycle ralentit la contraction sans l'empêcher. Et le point fixe est remarquable : `Cat(a) = Cat(b) = 0,618034`, valeur qui coïncide exactement avec **(√5−1)/2**, l'inverse du nombre d'or.

Ce n'est pas un hasard numérique : la récurrence s'écrit `x = 1/(1+x)` quand `a` et `b` se valent (chacun a exactement un attaquant, de même force). En multipliant par `(1+x)`, on obtient `x² + x − 1 = 0`, dont la racine positive est précisément `(√5−1)/2 ≈ 0,618034`. La sortie affiche la valeur théorique à côté de la valeur calculée — l'accord à six décimales **vérifie l'implémentation** du point fixe.

**Le verdict partagé** : sur un cycle symétrique, h-Categoriser **ne crée pas** d'ordre que Dung aurait manqué — les deux arguments restent à égalité stricte (0,618034 chacun). La différence entre les deux sémantiques ne porte donc pas sur l'existence d'un ordre, mais sur sa **finesse** : tout deux-capables quand les défaites sont symétriques, départage seulement quand le nombre d'attaquants diffère (le cas `a` contre `b` de la section 3).

## 6. Exercices

> Les cellules ci-dessous sont des **squelettes à compléter**. Le notebook s'exécute de bout en bout
> même si vous ne les complétez pas (elles renvoient `None` par défaut).

### Exercice 1 — Départage strict

Complétez `is_strictly_stronger(cat, a, b)` : renvoyer `Some(true)` si `a` est **strictement** mieux classé
que `b` selon le vecteur `cat` de h-Categoriser, `Some(false)` sinon.

In [8]:
use std::collections::{BTreeMap, BTreeSet};

/// Cadre d'argumentation abstrait de Dung : <args, attacks>.
struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    /// Arguments qui attaquent directement x.
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        let mut out = BTreeSet::new();
        for (a, b) in self.attacks.iter() {
            if *b == x {
                out.insert(*a);
            }
        }
        out
    }
}

/// Calcule le classement h-Categoriser par iteration de point fixe.
fn h_categoriser(af: &Af, tol: f64, max_iter: usize) -> (BTreeMap<&'static str, f64>, usize) {
    let mut cat: BTreeMap<&'static str, f64> = af.args.iter().map(|a| (*a, 1.0)).collect();
    for it in 1..=max_iter {
        let mut nxt: BTreeMap<&'static str, f64> = BTreeMap::new();
        for a in af.args.iter() {
            let somme: f64 = af.attackers(a).iter().map(|b| cat[*b]).sum();
            nxt.insert(*a, 1.0 / (1.0 + somme));
        }
        let mut delta = 0.0_f64;
        for a in af.args.iter() {
            let ecart = (nxt[*a] - cat[*a]).abs();
            if ecart > delta {
                delta = ecart;
            }
        }
        cat = nxt;
        if delta < tol {
            return (cat, it);
        }
    }
    (cat, max_iter)
}

/// Departage strict : Some(true) si a est strictement mieux classe que b.
fn is_strictly_stronger(
    _cat: &BTreeMap<&str, f64>,
    _a: &str,
    _b: &str,
) -> Option<bool> {
    // TODO : comparer cat[a] et cat[b]
    // Indice : "strictement mieux classe" = force strictement superieure.
    None // remplacez par votre comparaison
}

/// Booleen au format Python (True / False), et None pour un stub non fait.
fn py_opt(v: Option<bool>) -> String {
    match v {
        Some(b) => py_bool(b).to_string(),
        None => "None".to_string(),
    }
}

/// Booleen au format Python (True / False).
fn py_bool(b: bool) -> &'static str {
    if b {
        "True"
    } else {
        "False"
    }
}

// Exemple fil rouge : e n'est pas ataque, a subit 1 attaque, b en subit 3.
let af = Af {
    args: ["e", "a", "b", "y", "z1", "z2", "z3"].into_iter().collect(),
    attacks: [("y", "a"), ("z1", "b"), ("z2", "b"), ("z3", "b")].into_iter().collect(),
};
let (cat, _n_iter) = h_categoriser(&af, 1e-12, 10_000);

// Test attendu (une fois complete) : True, puis False.
println!("a strictement > b ? {}", py_opt(is_strictly_stronger(&cat, "a", "b")));
println!("b strictement > a ? {}", py_opt(is_strictly_stronger(&cat, "b", "a")));

a strictement > b ? None


b strictement > a ? None


### Exercice 2 — Construire un cas de départage

Définissez un cadre `af_ex2` à **4 arguments** dans lequel h-Categoriser **départage** deux arguments que
la sémantique fondée déclare tous deux rejetés (OUT).

*Indice : reprenez le motif de la section 3 — un argument attaqué une fois, un autre attaqué deux fois,
par des arguments non attaqués.*

In [9]:
use std::collections::{BTreeMap, BTreeSet};

/// Cadre d'argumentation abstrait de Dung : <args, attacks>.
struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    /// Arguments qui attaquent directement x.
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        let mut out = BTreeSet::new();
        for (a, b) in self.attacks.iter() {
            if *b == x {
                out.insert(*a);
            }
        }
        out
    }
}

/// Calcule le classement h-Categoriser par iteration de point fixe.
fn h_categoriser(af: &Af, tol: f64, max_iter: usize) -> (BTreeMap<&'static str, f64>, usize) {
    let mut cat: BTreeMap<&'static str, f64> = af.args.iter().map(|a| (*a, 1.0)).collect();
    for it in 1..=max_iter {
        let mut nxt: BTreeMap<&'static str, f64> = BTreeMap::new();
        for a in af.args.iter() {
            let somme: f64 = af.attackers(a).iter().map(|b| cat[*b]).sum();
            nxt.insert(*a, 1.0 / (1.0 + somme));
        }
        let mut delta = 0.0_f64;
        for a in af.args.iter() {
            let ecart = (nxt[*a] - cat[*a]).abs();
            if ecart > delta {
                delta = ecart;
            }
        }
        cat = nxt;
        if delta < tol {
            return (cat, it);
        }
    }
    (cat, max_iter)
}

// TODO : construire af_ex2 = Af({...}, {...})
let af_ex2: Option<Af> = None; // remplacez par votre cadre

if let Some(af2) = af_ex2 {
    let (c2, _n) = h_categoriser(&af2, 1e-12, 10_000);
    let forces: Vec<String> = c2
        .iter()
        .map(|(a, v)| format!("{}: {:.3}", a, v))
        .collect();
    println!("Forces h-Cat : {{{}}}", forces.join(", "));
} else {
    println!("Exercice 2 a completer");
};

Exercice 2 a completer


### Exercice 3 — Détecter la convergence du fardeau

Complétez `burden_converged(af, steps, tol)` : renvoyer `Some(true)` si, sur l'AF donné, le **dernier** pas
de la suite de fardeau ne diffère du précédent que de moins de `tol` pour **tous** les arguments.

*Indice : `burden_numbers(af, steps)` renvoie les vecteurs ; comparez les deux dernières composantes.*

In [10]:
use std::collections::{BTreeMap, BTreeSet};

/// Cadre d'argumentation abstrait de Dung : <args, attacks>.
struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    /// Arguments qui attaquent directement x.
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        let mut out = BTreeSet::new();
        for (a, b) in self.attacks.iter() {
            if *b == x {
                out.insert(*a);
            }
        }
        out
    }
}

/// Renvoie {a: (Bur^1, ..., Bur^steps)} (Bur^0 = 1 pour tous).
fn burden_numbers(af: &Af, steps: usize) -> BTreeMap<&'static str, Vec<f64>> {
    let mut bur: BTreeMap<&'static str, f64> = af.args.iter().map(|a| (*a, 1.0)).collect();
    let mut seq: BTreeMap<&'static str, Vec<f64>> =
        af.args.iter().map(|a| (*a, Vec::new())).collect();
    for _ in 0..steps {
        let mut nxt: BTreeMap<&'static str, f64> = BTreeMap::new();
        for a in af.args.iter() {
            let somme: f64 = af.attackers(a).iter().map(|b| 1.0 / bur[*b]).sum();
            nxt.insert(*a, 1.0 + somme);
        }
        for a in af.args.iter() {
            seq.get_mut(a).unwrap().push(nxt[*a]);
        }
        bur = nxt;
    }
    seq
}

/// Convergence du fardeau : Some(true) si le dernier pas differe de moins de tol.
fn burden_converged(_af: &Af, _steps: usize, _tol: f64) -> Option<bool> {
    // TODO : comparer les deux derniers pas de chaque vecteur de fardeau
    // bn = burden_numbers(af, steps) ; bn[a] est un vecteur de longueur steps.
    None // remplacez par votre test de convergence
}

/// Booleen au format Python (True / False), et None pour un stub non fait.
fn py_opt(v: Option<bool>) -> String {
    match v {
        Some(b) => py_bool(b).to_string(),
        None => "None".to_string(),
    }
}

/// Booleen au format Python (True / False).
fn py_bool(b: bool) -> &'static str {
    if b {
        "True"
    } else {
        "False"
    }
}

// Exemple fil rouge : e n'est pas attaque, a subit 1 attaque, b en subit 3.
let af = Af {
    args: ["e", "a", "b", "y", "z1", "z2", "z3"].into_iter().collect(),
    attacks: [("y", "a"), ("z1", "b"), ("z2", "b"), ("z3", "b")].into_iter().collect(),
};
let _bn = burden_numbers(&af, 8);

println!(
    "Fardeau convergent sur l'AF fil rouge ? {}",
    py_opt(burden_converged(&af, 8, 1e-6))
);

Fardeau convergent sur l'AF fil rouge ? None


## À retenir

- Les sémantiques **de classement** raffinent le tout-ou-rien de Dung en attribuant une **force
  numérique** à chaque argument, ce qui **départage** des arguments de même statut — mesuré sur le fil
  rouge : `a` et `b` tous deux OUT pour Dung, mais `Cat(a) = 0.5000` contre `Cat(b) = 0.2500`.
- **h-Categoriser** ($\mathrm{Cat}(a) = 1/(1+\sum_{b\to a}\mathrm{Cat}(b))$) est un point fixe qui
  **converge** sur tout AF fini, y compris cyclique.
- La sémantique du **fardeau** classe par comparaison **lexicographique** d'une suite de nombres.
- Les deux respectent des **principes** structurels (abstraction, indépendance, contre-transitivité).

## Vérifiez votre compréhension

**Question 1 — Pourquoi la sortie de la section 3 imprime-t-elle deux `True` plutôt qu'un seul ?**

La cellule imprime `Meme statut Dung (a, b tous deux OUT) ? True` puis `h-Categoriser les departe (Cat(a) > Cat(b)) ? True`.

<details><summary>Réponse</summary>

Parce que ce sont **deux propositions différentes** : la première dit que la sémantique fondée de Dung ne distingue pas `a` (attaqué par 1 argument) de `b` (attaqué par 3) — tous deux OUT ; la seconde dit que h-Categoriser, lui, les départage (`Cat(a) = 0.5` contre `Cat(b) = 0.25`). Ensemble, les deux `True` **mesurent la limite de Dung** : c'est le contraste exact entre statut tout-ou-rien et force graduée — et pourquoi les sémantiques de classement existent.

</details>

**Question 2 — Le classement par fardeau rend `a -> (2.000, 2.000, 2.000)` et `b -> (4.000, 4.000, 4.000)` : que signifie le triplet, et pourquoi suffit-il à départager ?**

La section 4 imprime le classement complet `Bur^1..3` pour les sept arguments du fil rouge, et la cellule d'accord conclut `h-Cat : a avant b ? True` / `Fardeau : a avant b ? True`.

<details><summary>Réponse</summary>

Chaque composante `Bur^k` compte les chaînes d'attaques de longueur `k` aboutissant à l'argument (fardeau de degré 1, 2, 3) : `b` porte le double du fardeau de `a` à **tous** les degrés. La comparaison est **lexicographique** — on départage au premier degré où les suites diffèrent — et l'accord imprimé avec h-Cat (`a avant b` des deux côtés) montre deux échelles numériques différentes, un même ordre : une convergence de méthodes, pas une tautologie.

</details>

**Question 3 — Sur le cycle a ↔ b, pourquoi citer la valeur théorique $(\sqrt{5}-1)/2$ à côté de la valeur mesurée ?**

La section 5 imprime `AF cyclique a<->b : convergence en 29 iterations`, `Cat(a) = Cat(b) = 0.618034`, `Valeur theorique (sqrt(5)-1)/2 = 0.618034`, et `Statut Dung (fonde) des deux : UNDEC`.

<details><summary>Réponse</summary>

Parce que le cycle donne une équation de point fixe **résoluble analytiquement** : Cat = 1/(1+Cat) a pour solution l'inverse du nombre d'or. La valeur mesurée à 10⁻⁶ près après 29 itérations prouve la convergence vers ce point fixe — la théorie prédit, l'itération vérifie. Et le contraste est complet : là où Dung rend les deux arguments UNDEC (extension fondée vide), le classement rend une force **finie et égale** pour chacun. Les trois exercices prolongent ce travail : départage strict, cas de départage à construire, détection de convergence du fardeau.

</details>

## Pour aller plus loin

- [Version Python de ce carnet](Argumentation-03c-Ranking-Semantics-Python.ipynb) — le jumeau Python (memes instances, memes oracles).
- [Index de la sous-série ArgumentAnalysis](README.md) — la place des sémantiques de classement dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet, du raisonnement formel aux systèmes multi-agents.

La bibliothèque **Tweety** (`org.tweetyproject.arg.rankings`) implémente ces sémantiques et bien d'autres (Categoriser, Burden, Discussion-based, Tuples\*, Matt & Toni). Voir aussi le notebook compagnon [`Argumentation-03-Dung-AF-Semantics`](Argumentation-03-Dung-AF-Semantics-Python.ipynb) pour les sémantiques d'extension, et [`Argumentation-03b-Value-Based-AF`](Argumentation-03b-Value-Based-AF-Python.ipynb) pour une autre voie de départage (préférences d'audience plutôt que force numérique).

**Références.**
- P. Besnard, A. Hunter (2001). *A logic-based theory of deductive arguments*. Artificial Intelligence.
- L. Amgoud, J. Ben-Naim (2013). *Ranking-based semantics for argumentation frameworks*. SUM.
- F. Pu, J. Luo, Y. Zhang, G. Luo (2014). *Argument ranking with categoriser function*. KSEM (convergence).